# Configuración del ecosistema

Este notebook recorre la configuración completa del entorno para el pipeline de análisis.

**No hace falta para seguir el curso.** Todos los resultados que usan los notebooks están precomputados en `results/`. Ejecuta esto solo si más adelante vas a recalcular embeddings/NER por tu cuenta (por ejemplo con `scripts/precompute.py` sobre otros datos) y quieres comprobar antes que tu instalación de Ollama y GPU funciona — descarga ambos modelos (~9GB) y corre una prueba en vivo.

**Qué vamos a ver:**
1. Dependencias de Python
2. Modelos de Ollama (embeddings + LLM)
3. Comprobar que se usa la GPU
4. Prueba del pipeline completo (parsear → generar embeddings → comparar)

## 1. Dependencias de Python

In [ ]:
# Install all dependencies
%pip install -q \
    ollama \
    pandas \
    numpy \
    matplotlib \
    seaborn \
    plotly \
    scikit-learn \
    umap-learn \
    langdetect \
    pytz \
    tqdm

## 2. Modelos de Ollama

Usamos dos modelos:
- **`qwen3-embedding`** — embeddings rápidos y de buena calidad. Se usan para estilometría y correlación de usuarios cross-foro.
- **`qwen2.5:14b`** — LLM para detección de idioma, extracción de entidades y clasificación de posts. El de 14B cabe en 12GB de VRAM.

In [ ]:
import subprocess

models = ["qwen3-embedding", "qwen2.5:14b"]

for model in models:
    print(f"Descargando {model}...")
    subprocess.run(["ollama", "pull", model], check=True)
    print(f"  ✓ {model} listo\n")

## 3. Comprobar uso de GPU

Ollama acelera por GPU en NVIDIA (CUDA), AMD (ROCm/Vulkan) y Apple (Metal) — no es exclusivo de NVIDIA. El chequeo de abajo intenta primero `nvidia-smi` y, si no existe, `rocm-smi` (AMD). Si tu GPU no aparece con ninguna de las dos herramientas, no significa que no se esté usando: la fuente fiable es el log del servidor de Ollama (busca `library=cuda`, `library=rocm` o `library=vulkan` al cargar el modelo).

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
        capture_output=True, text=True,
    )
    print(result.stdout)
elif shutil.which("rocm-smi"):
    result = subprocess.run(
        ["rocm-smi", "--showproductname", "--showmeminfo", "vram"],
        capture_output=True, text=True,
    )
    print(result.stdout)
else:
    print(
        "No se encontró nvidia-smi ni rocm-smi. Puede que no haya GPU, o que Ollama "
        "la use igualmente por otra vía (p. ej. Vulkan en AMD, Metal en Apple). "
        "Para confirmarlo, revisa el log del servidor de Ollama al cargar un modelo."
    )

In [ ]:
import ollama

# Tras descargar un modelo, comprobamos que se cargó en la GPU y no en CPU
response = ollama.embed(model="qwen3-embedding", input="test")
print(f"Dimensión del embedding: {len(response.embeddings[0])}")

if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"], capture_output=True, text=True)
    print(f"VRAM usada tras cargar el modelo (NVIDIA): {result.stdout.strip()}")
elif shutil.which("rocm-smi"):
    result = subprocess.run(["rocm-smi", "--showmeminfo", "vram"], capture_output=True, text=True)
    print(f"VRAM usada tras cargar el modelo (AMD):\n{result.stdout}")
else:
    print("No se pudo leer la VRAM con herramientas de fabricante — revisa el log del servidor de Ollama.")

## 4. Prueba del pipeline completo

Simula el pipeline real con datos sintéticos: parsear → generar embeddings → comparar (similitud coseno).

In [ ]:
import pandas as pd
import ollama
from tqdm import tqdm

# Posts sintéticos de foro
sample_posts = [
    {"user": "darkuser1", "text": "selling fresh cc dumps, 95% valid rate, pm me"},
    {"user": "darkuser2", "text": "looking for rdp access to us bank networks"},
    {"user": "darkuser1", "text": "new batch available, eu cards, contact via jabber"},
    {"user": "darkuser3", "text": "anyone has working method for cashout 2024?"},
]

df = pd.DataFrame(sample_posts)
df

In [ ]:
# Generar embeddings de todos los posts
embeddings = []
for text in tqdm(df["text"], desc="Generando embeddings"):
    response = ollama.embed(model="qwen3-embedding", input=text)
    embeddings.append(response.embeddings[0])

df["embedding"] = embeddings
print(f"Embeddings generados: {len(embeddings)} posts — dim={len(embeddings[0])}")

In [ ]:
# Búsqueda semántica — similitud coseno por fuerza bruta, mismo enfoque que scripts/precompute.py.
# Sin base de datos vectorial: a esta escala (unos pocos posts, o incluso el dataset completo),
# comparar la consulta contra cada embedding directamente es más simple y suficientemente rápido.
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = "credit card dumps for sale"
query_embedding = ollama.embed(model="qwen3-embedding", input=query).embeddings[0]

matrix = np.array(df["embedding"].tolist())
sims = cosine_similarity([query_embedding], matrix)[0]
top = sims.argsort()[::-1][:3]

print(f"Consulta: '{query}'\n")
for i in top:
    print(f"  [{df.iloc[i]['user']}] {df.iloc[i]['text']}  (similitud: {sims[i]:.3f})")

## Configuración completa

| Componente | Estado |
|-----------|--------|
| Dependencias Python | ✓ |
| qwen3-embedding | ✓ |
| qwen2.5:14b | ✓ |
| Inferencia por GPU | ✓ |

Listos para los casos prácticos.